[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/greencube-dev-boop/crypto-forecast/blob/main/notebooks/accuracy_eval.ipynb)

# 📈 Retrospective 48-Hour Accuracy Evaluation Pipeline

Generates and incrementally maintains accuracy analysis CSV datasets tracking real settled values against 1h-to-48h retrospective forecasts across 4 models (**TimesFM**, **Chronos**, **Granite TTM**, and **Persistence**) and configurable cryptocurrency pairs.

### Evaluation Protocol:
- **Zero Lookahead**: For each target timestamp $t$ and horizon $k \in [1, 48]$, historical context strictly ending at $t - k\text{h}$ is fed into the model.
- **Intermediate Step Discarding**: For horizon $k$, predictions for intermediate steps $1 \dots k - 1$ are ignored, and only the horizon $k$ prediction (for target time $t$) is recorded.
- **Google Drive Incremental Persistence**: Automatically saves to `/content/drive/MyDrive/Trader/accuracy_eval/`, detecting already evaluated hours and appending only newly completed timestamps across runs.
- **Headless Execution**: Strictly non-interactive (zero GUI dependencies).


In [ ]:
# 1. Hardware and Execution Device Check
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Active Execution Device: {device.upper()}")
if device == "cuda":
    print(f"[INFO] GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"[INFO] VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("[INFO] Running in CPU mode. Lightweight fallback engines active.")


In [ ]:
# 2. Google Drive Mount & Evaluation Directory Setup
import os
from pathlib import Path

DRIVE_ACCURACY_DIR = "/content/drive/MyDrive/Trader/accuracy_eval"
LOCAL_ACCURACY_DIR = "./accuracy_eval"

is_colab = "google.colab" in os.environ or os.path.exists("/content")
drive_mounted = False

if is_colab:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        drive_mounted = os.path.exists("/content/drive/MyDrive")
    except Exception as exc:
        print(f"[WARN] Drive mount skipped or failed: {exc}")

if drive_mounted:
    OUTPUT_DIR = DRIVE_ACCURACY_DIR
    print(f"[INFO] Google Drive mounted successfully. Target directory: {OUTPUT_DIR}")
else:
    OUTPUT_DIR = LOCAL_ACCURACY_DIR
    print(f"[INFO] Using local fallback directory: {OUTPUT_DIR}")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


In [ ]:
# 3. Synchronize Project Repository from GitHub (greencube-dev-boop)
import os
import sys

repo_url = "https://github.com/greencube-dev-boop/crypto-forecast.git"
repo_dir = "/content/crypto-forecast"

if os.path.exists("/content"):
    if not os.path.exists(repo_dir):
        print(f"[INFO] Cloning remote repository from {repo_url}...")
        !git clone {repo_url} {repo_dir}
        %cd {repo_dir}
    else:
        print(f"[INFO] Updating existing repository at {repo_dir}...")
        %cd {repo_dir}
        !git clean -fd
        !git fetch origin main
        !git reset --hard origin/main

if "src" not in sys.path:
    sys.path.insert(0, "src")
print(f"[INFO] Python sys.path configured. Current working directory: {os.getcwd()}")


In [ ]:
# 4. Install Dependencies
!pip install -r requirements.txt -q
print("[INFO] Dependencies verified and installed.")


In [ ]:
# 5. Declarative Evaluation Configuration
PAIRS = [
    "BTC/USDT",
    "ETH/USDT",
    "SOL/USDT",
    "BNB/USDT",
]

MODELS = [
    "timesfm",
    "chronos",
    "granite",
    "persistence",
]

MAX_HORIZONS = 48
CONTEXT_WINDOW = 512
INITIAL_BACKFILL_HOURS = 48

print(f"[CONFIG] Pairs ({len(PAIRS)}): {PAIRS}")
print(f"[CONFIG] Models ({len(MODELS)}): {MODELS}")
print(f"[CONFIG] Retrospective Horizons: 1h to {MAX_HORIZONS}h")
print(f"[CONFIG] Output Directory: {OUTPUT_DIR}")


In [ ]:
# 6. Execute Retrospective Accuracy Evaluation Pipeline (Headless)
from crypto_forecast.eval import run_accuracy_evaluation

eval_results = run_accuracy_evaluation(
    pairs=PAIRS,
    models=MODELS,
    output_dir=OUTPUT_DIR,
    initial_backfill_hours=INITIAL_BACKFILL_HOURS,
    max_horizon=MAX_HORIZONS,
    context_window=CONTEXT_WINDOW,
    force_fallback=False,
    verbose=True,
)

print(f"\n[DONE] Pipeline execution completed. {len(eval_results)} CSV files processed.")


In [ ]:
# 7. Headless Dataset Summary & Verification Table
import pandas as pd
from pathlib import Path

summary_rows = []
out_path = Path(OUTPUT_DIR)

for key, csv_file in sorted(eval_results.items()):
    if csv_file.exists() and csv_file.stat().st_size > 0:
        df = pd.read_csv(csv_file)
        size_kb = csv_file.stat().st_size / 1024
        summary_rows.append({
            "Dataset": csv_file.name,
            "Rows": len(df),
            "Columns": df.shape[1],
            "Earliest Time": df["real time"].min() if "real time" in df.columns else "N/A",
            "Latest Time": df["real time"].max() if "real time" in df.columns else "N/A",
            "Size (KB)": f"{size_kb:.1f}",
        })
    else:
        summary_rows.append({
            "Dataset": csv_file.name,
            "Rows": 0,
            "Columns": 0,
            "Earliest Time": "N/A",
            "Latest Time": "N/A",
            "Size (KB)": "0.0",
        })

summary_df = pd.DataFrame(summary_rows)
print("\n" + "=" * 80)
print("ACCURACY EVALUATION DATASET INVENTORY")
print("=" * 80)
print(summary_df.to_string(index=False))

# Preview the latest rows of the first non-empty dataset
for key, csv_file in eval_results.items():
    if csv_file.exists() and csv_file.stat().st_size > 0:
        df_sample = pd.read_csv(csv_file)
        print(f"\n[Preview] Last 3 rows from {csv_file.name}:")
        preview_cols = ["real time", "value at time", "value predicted 1h before", "value predicted 24h before", "value predicted 48h before"]
        available_preview_cols = [c for c in preview_cols if c in df_sample.columns]
        print(df_sample[available_preview_cols].tail(3).to_string(index=False))
        break
